# Many-Body Fractional Charge Pump
## One-dimensional flux-cylinder diagnostic for interacting topological phases

**Abstract.** This notebook is a self-contained pedagogical derivation and hands-on guide to the
**fractional charge pump** implemented in `observables/charge_pump.py`. The calculation threads a
flux $\theta$ through one periodic direction of a finite torus and tracks the winding of the
many-body polarization in the transverse direction — a finite-size analogue of the
Laughlin–Thouless charge pump. It is deliberately *not* a two-dimensional many-body Chern-number
calculation (which would need a full 2D flux-torus grid); the 1D pump is faster, more intuitive,
and captures the same universal fractional Hall response. The key design innovation is the
**flux-aware symmetry group**: the calculation stays in symmetry-resolved momentum sectors whose
labels remain invariant under flux insertion, giving orders-of-magnitude speedup.

**References.**
1. R. Resta, *Quantum-mechanical position operator in extended systems*, Phys. Rev. Lett. **80**, 1800 (1998).
2. R. B. Laughlin, *Quantized Hall conductivity in two dimensions*, Phys. Rev. B **23**, 5632 (1981).
3. D. J. Thouless, *Quantization of particle transport*, Phys. Rev. B **27**, 6083 (1983).
4. D. N. Sheng, Z.-C. Gu, K. Sun, L. Sheng, *Fractional Chern insulator on the honeycomb lattice with bosons*, Phys. Rev. Lett. **107**, 146803 (2011).
5. K. Sun, Z. Gu, H. Katsura, S. Das Sarma, *Nearly flatbands with nontrivial topology*, Phys. Rev. Lett. **106**, 236803 (2011).
6. XDiag — the symmetry-resolved ED framework whose orbit–stabilizer design philosophy we follow: [github.com/awietek/xdiag](https://github.com/awietek/xdiag).


## Physical Setup: Flux Cylinder and Laughlin Pump

Consider a two-dimensional periodic finite system (a torus) with $L_x\times L_y$ unit cells. We
thread an adiabatic flux $\theta(t)$ along the $x$-direction. In the non-interacting integer quantum
Hall regime, Laughlin's gauge argument shows that each occupied band with Chern number $C$ pumps $C$
charges across any $y$-section per flux quantum. For strongly correlated fractional phases the same
flux-threading pumps a *fractional* charge — the many-body Hall conductance in units of $e^2/h$.

- **Flux insertion**: $\theta_x$ multiplies the hoppings crossing the $x$-periodic boundary by a
  Peierls phase $e^{2\pi i\theta_x}$.
- **Polarization measurement**: in the transverse ($y$) direction we define the periodic many-body
  position operator following Resta,

\begin{equation}
\boxed{\hat U_y = \exp\Big(\frac{2\pi i}{L_y}\sum_{j=1}^{N} x_{j,y}\,\hat n_j\Big),}
\end{equation}

where $x_{j,y}$ is the crystal coordinate of site $j$ along $y$ in unit-cell units (including the
sublattice offset $\delta_y(\mathrm{sub}_j)$ by default), and $L_y$ the number of unit cells in that
direction.

$\hat U_y$ is unitary and periodic: $\hat U_y^{L_y}=\mathbb 1$. Its eigenvalues are phases
$e^{2\pi iP}$ where $P\in[0,1)$ is the **many-body polarization**. The charge pumped across a
$y$-cut when $\theta_x$ advances by one flux quantum is

\begin{equation}
\boxed{\Delta Q = P(\theta_x=1)-P(\theta_x=0)\qquad(\text{in units of }e).}
\end{equation}

For a fractional Chern insulator at filling $\nu=p/q$ there are $q$ nearly-degenerate ground states
on the torus (the topological multiplet). Threading one flux quantum cyclically permutes them, and
each carries a polarization winding of $\Delta Q = p/q$. The total pumped charge $\sum_i\Delta Q_i$
is an integer — the many-body Chern number.


## Flux-Aware Symmetry: Why Momentum Labels Stay Fixed

Under flux insertion the Hamiltonian $H(\theta)$ does **not** commute with the ordinary translation
$T^0$ — the boundary-gauge Peierls phase breaks naive translation invariance. The correct
translation that *does* commute with $H(\theta)$ is the **gauge-covariant translation**

\begin{equation}
T^{\theta}_{(d_x,d_y)} = G_\theta\, T^0_{(d_x,d_y)}\, G_\theta^{-1},\qquad
G_\theta = \exp\Big(i 2\pi \sum_j \frac{\theta\cdot x_j}{L}\,\hat n_j\Big).
\end{equation}

$T^\theta$ is unitarily equivalent to $T^0$, so it has the **same** eigenvalue spectrum — the
standard crystal momenta $[k_1,k_2]$ — and the irrep labels are unchanged under flux. In the
implementation, `build_translation_group(lattice, θ)` absorbs the gauge transformation into the
`perm_phases` field of each `Symmetry_Operation`. The orbit *partition* (which Fock states belong to
which orbit) depends only on the permutation part and is therefore invariant; only the stabilizer
phases need updating at each $\theta$ (`update_orbit_stabilizer_phases`), avoiding a full
$O(\binom{N}{N_e})$ Gosper enumeration per flux point.


## Symmetry-Resolved Projected Position Operator

Because $\hat U_y$ is **diagonal** in the Fock basis, $\hat U_y|m\rangle=u_y(m)|m\rangle$ with

\begin{equation}
u_y(m) = \prod_{j\in\mathrm{occ}(m)} e^{2\pi i x_{j,y}/L_y},
\end{equation}

its matrix elements between symmetry-projected basis states factorize cleanly. Recall (from
`design.ipynb`) the normalized symmetry-projected state for orbit representative $[s]$ and irrep
$\chi$:

\begin{equation}
|\widetilde{[s];\chi}\rangle = \sqrt{\frac{|\mathrm{Stab}(s)|}{|G|}}\sum_{g\in G/\mathrm{Stab}(s)} \chi(g)^*\, U_g |[s]\rangle .
\end{equation}

For a diagonal operator $D|m\rangle=d(m)|m\rangle$, the matrix element between irreps
$\chi_{\mathrm{to}}$ and $\chi_{\mathrm{from}}$ on the **same** orbit representative is

\begin{equation}
\boxed{\langle \widetilde{[s];\chi_{\mathrm{to}}}|D|\widetilde{[s];\chi_{\mathrm{from}}}\rangle
= \frac{1}{|G|}\sum_{g\in G}\chi_{\mathrm{to}}(g)\,\chi_{\mathrm{from}}(g)^*\,d(g\cdot s).}
\end{equation}

Representatives from different orbits give exactly zero (a diagonal operator cannot connect disjoint
orbits). The prefactor $1/|G|$ (rather than $\sqrt{\mathrm{Stab}}$ factors) comes from the
normalization-convention cancellation derived in `design.ipynb`. The implementation
(`_position_operator_matrix`) builds this inter-sector representation as a sparse matrix and then
projects it into the low-energy eigenbasis of $H(\theta)$.


## Algorithmic Walkthrough

At each flux value $\theta$, `flux_charge_pump` performs the following steps:

1. **Update the model in place** — `update_second_quantized_model_with_twisted_phases(model,
   twisted_phases_over_2π=[θ, 0])` regenerates the bilinear terms with the Peierls phase.
2. **Rebuild the flux-aware symmetry group** — `build_translation_group(model.lattice, [θ, 0])`;
   the orbit partition is reused, only stabilizer phases are updated.
3. **Diagonalize the requested sectors** — via the sparse symmetry-resolved block (ARPACK) at each
   flux point, cached as per-$\theta$ checkpoints.
4. **Project $\hat U_y$ into the low-energy manifold** —
   $P_{\alpha\beta}(\theta)=\sum_{a,b}(\psi^{(\alpha)}_a)^\dagger U_y^{\alpha\beta}\psi^{(\beta)}_b$,
   where $\psi^{(\alpha)}_a$ is the $a$-th low-energy eigenvector in sector $\chi_\alpha$.
5. **Unwrap phases and extract the pumped charge** — the eigenvalues $\lambda_i(\theta)$ of $P(\theta)$
   are complex phases; we sort by argument at each $\theta$ and unwrap the branches with optimal
   permutation matching to minimize discontinuities, then
   $\Delta Q_i = P_i(\theta_{\max})-P_i(0)$.

The result stores `polarizations` (raw unwrapped phases), `pumped_charge_trajectories` (shifted to
start from zero, for plotting), and `pumped_charges` (the final winding of each branch).


## Hands-On: Bosonic Haldane FCI on $[2,3]$ Honeycomb ($\Delta Q\approx 0.5$)

The Haldane honeycomb model at half filling of the lower Chern band hosts the bosonic FCI at
$\nu=1/2$ (Sheng *et al.*). On the $2\times3$ torus (12 sites, 3 hard-core bosons) the ground state
is the finite-size bosonic Laughlin state with topological ground-state degeneracy $2$ at momentum
sectors $[0,0]$ and $[1,0]$. Threading one flux quantum should advance each polarization branch by
**half a charge**.


In [ ]:
import os
from fractions import Fraction
import numpy as np
import realspace_exactdiagonalization_py as ed

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(ed.__file__)))
FIG_DIR = os.path.join(PROJECT_ROOT, "doc", "figures")
CKPT_DIR = os.path.join(PROJECT_ROOT, "doc", "checkpoints")
os.makedirs(FIG_DIR, exist_ok=True)

sample_size = [2, 3]
model = ed.build_zero_flux_bosonic_fci_second_quantized_model(
    sample_size=sample_size, params=ed.params_DNSheng)
labels = ed.default_fci_sectors(sample_size)          # [(0,0), (1,0)]
flux_list = list(np.linspace(0.0, 1.0, 7))           # 7 flux points (default 9)

pump = ed.flux_charge_pump(
    model, labels,
    filling_fraction=Fraction(1, 4),   # 3 bosons / 12 vertices
    flux_direction=1,                  # thread θ_x
    polarization_direction=2,          # measure U_y polarization
    twisted_phases_over_2π_list=flux_list,
    nev_per_sector=1,
    fig_path=os.path.join(FIG_DIR, "bosonic_FCI_charge_pump_23.svg"),
    checkpoint_dir=CKPT_DIR,
)

print("pumped charges ΔQ =", pump.pumped_charges)
print("|ΔQ| per branch ≈ 0.5 → total =", round(abs(pump.pumped_charges).sum(), 6))


| Polarization branch | $\Delta Q$ (pumped charge) |
|---|---|
| Branch 1 (sector $[0,0]$) | $\approx 0.5$ |
| Branch 2 (sector $[1,0]$) | $\approx 0.5$ |
| **Total** | $\mathbf{1.0}$ (integer) |

This is the **semion topological order** — the finite-size fingerprint of the $\nu=1/2$ bosonic
Laughlin state. The spectrum-flow observable sees the same physics as a cyclic exchange of the two
ground states: the $[0,0]$ ground state adiabatically evolves into the $[1,0]$ ground state after
one flux quantum and returns after two.


## Hands-On: Fermionic Checkerboard FCI on $[3,4]$ ($\Delta Q\approx 2/3$)

The SGKS checkerboard model with two flat Chern bands hosts a fermionic FCI at $\nu=2/3$ filling of
the lower band (8 spinless fermions on the $3\times4\times2=24$ flattened vertices). Its three
nearly-degenerate ground states live at $k=(0,0),(1,0),(2,0)$ (GSD$=3$); threading one flux quantum
cyclically permutes them and each polarization branch winds by **two-thirds of a charge**.


In [ ]:
import os
from fractions import Fraction
import numpy as np
import realspace_exactdiagonalization_py as ed

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(ed.__file__)))
FIG_DIR = os.path.join(PROJECT_ROOT, "doc", "figures")
CKPT_DIR = os.path.join(PROJECT_ROOT, "doc", "checkpoints")
os.makedirs(FIG_DIR, exist_ok=True)

sample_size = [3, 4]
model = ed.build_zero_flux_fermionic_fci_second_quantized_model(
    sample_size=sample_size, params=ed.params_Sun_Gu_Katsura_Sarma)
labels = ed.default_fci_sectors_fermionic(sample_size)   # [(0,0),(1,0),(2,0)]
flux_list = list(np.linspace(0.0, 1.0, 7))

pump = ed.flux_charge_pump(
    model, labels,
    filling_fraction=Fraction(1, 3),   # 8 fermions / 24 vertices (ν=2/3 per band)
    flux_direction=1,
    polarization_direction=2,
    twisted_phases_over_2π_list=flux_list,
    nev_per_sector=1,
    fig_path=os.path.join(FIG_DIR, "fermionic_FCI_charge_pump_34_nu23.svg"),
    checkpoint_dir=CKPT_DIR,
)

print("pumped charges ΔQ =", pump.pumped_charges)
print("|ΔQ| per branch ≈ 2/3 → total =", round(abs(pump.pumped_charges).sum(), 6))


The three branches each pump $\Delta Q\approx 2/3$, summing to $2$ — the integer many-body
Chern number of the $\nu=2/3$ fermionic FCI. (The particle–hole partner at $\nu=1/3$ pumps $1/3$ per
branch, total $1$; see `fermionic_fci.ipynb`.) The pump is the definitive finite-size diagnostic:
if the manifold were a trivial product state or a gapless liquid, the polarization would not wind by
a sharply quantized fraction.

---

*This notebook is part of `realspace_exactdiagonalization_py`. The core method is
`flux_charge_pump` in `observables/charge_pump.py`, with self-contained drivers
`test_bosonic_fci_charge_pump` and `test_fermionic_fci_charge_pump` in the `models` subpackage.*
